# TruthShield 00 — Setup, data download & HAVIC zero-shot baseline
Runtime → Change runtime type → **T4 GPU**. Run top to bottom. Safe to re-run: every step skips work already saved on Drive.

What this notebook does:
1. Mounts Drive, clones HAVIC + TruthShield code
2. Downloads HiFi-AVDF (~6.5 GB) and HAVIC weights (~860 MB) to Drive **once**
3. Unzips videos to fast local disk (`/content/hifi`) each session
4. Builds leakage-free pair-level splits
5. Verifies the HAVIC checkpoint loads with **zero missing keys**
6. Runs the official HAVIC sliding-window inference on the test split → baseline row of the ablation table

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
ROOT = '/content/drive/MyDrive/TruthShield'
for d in ['data/HiFi-AVDF', 'weights', 'splits', 'results', 'cache']:
    os.makedirs(f'{ROOT}/{d}', exist_ok=True)

# TruthShield code (public GitHub repo - no token needed)
TS_REPO = 'https://github.com/Jawad374/TruthShield.git'
# (Set TS_REPO = '' to instead use a copy of the code in Drive at ROOT/code.)
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
%cd /content
if not os.path.exists('/content/HAVIC'):
    !git clone -q --depth 1 https://github.com/tuffy-studio/HAVIC /content/HAVIC
# Patch HAVIC for current timm: new timm's Attention has no 'qk_scale' argument.
# HAVIC always passes qk_scale=None (= default scale), so dropping it changes nothing.
_f = '/content/HAVIC/src/models/interaction_modules.py'
_s = open(_f, encoding='utf-8').read()
_s = _s.replace('qkv_bias=qkv_bias, qk_scale=qk_scale, attn_drop=attn_drop, proj_drop=drop)',
                'qkv_bias=qkv_bias, attn_drop=attn_drop, proj_drop=drop)')
open(_f, 'w', encoding='utf-8').write(_s)
print('HAVIC timm patch applied:', 'qk_scale=qk_scale, attn_drop' not in _s)

if TS_REPO:
    if not os.path.exists('/content/TruthShield'):
        !git clone -q {TS_REPO} /content/TruthShield
    !git -C /content/TruthShield pull -q
    TS = '/content/TruthShield'
else:
    TS = f'{ROOT}/code'
assert os.path.exists(f'{TS}/src/models/dynamic_fusion.py'), f'TruthShield code not found at {TS}'

# Only what inference needs. Do NOT install HAVIC's requirements.txt (pins torch 2.3 / numpy 1.24 and breaks Colab).
!pip install -q decord ffmpeg-python einops timm huggingface_hub
# FaceX-Zoo torch.load()s whole pickled models; torch>=2.6 blocks that by default.
os.environ['TORCH_FORCE_NO_WEIGHTS_ONLY_LOAD'] = '1'

## Download dataset + weights to Drive (one time)

In [ ]:
from huggingface_hub import snapshot_download, hf_hub_download
snapshot_download('JielunPeng/HiFi-AVDF', repo_type='dataset', local_dir=f'{ROOT}/data/HiFi-AVDF')
W = hf_hub_download('JielunPeng/HAVIC', 'best_ft_model.pth', local_dir=f'{ROOT}/weights')
!ls -lh {ROOT}/data/HiFi-AVDF {ROOT}/weights

## Unzip to local disk (every session — reading video from Drive is slow)

In [ ]:
import glob, zipfile
LOCAL = '/content/hifi'
os.makedirs(LOCAL, exist_ok=True)
for z in sorted(glob.glob(f'{ROOT}/data/HiFi-AVDF/*.zip')):
    eng = os.path.basename(z)[:-4]
    if os.path.exists(f'{LOCAL}/.done_{eng}'):
        continue
    print('unzipping', eng)
    with zipfile.ZipFile(z) as f:
        f.extractall(LOCAL)
    open(f'{LOCAL}/.done_{eng}', 'w').close()

# Map CSV paths (e.g. 'sora2/1234.mp4') to real files, whatever the zip layout is.
mp4s = glob.glob(f'{LOCAL}/**/*.mp4', recursive=True)
by_rel = {}
for p in mp4s:
    parts = p.split('/')
    by_rel[parts[-2] + '/' + parts[-1]] = p
print(len(mp4s), 'videos on local disk')

## Leakage-free splits (pair-level, stratified by engine) + leave-one-engine-out

In [ ]:
import sys, pandas as pd

def use_src(root):
    """HAVIC and TruthShield both have a top-level `src` package.
    Drop any cached `src` modules and put `root` first so `import src...` resolves there."""
    for k in [k for k in sys.modules if k == 'src' or k.startswith('src.')]:
        del sys.modules[k]
    for p in ('/content/HAVIC', TS):
        while p in sys.path:
            sys.path.remove(p)
    sys.path.insert(0, root)

!cd {TS} && python -m src.data.make_splits --data_root {ROOT}/data/HiFi-AVDF --out_dir {ROOT}/splits
split = pd.read_csv(f'{ROOT}/splits/hifi_main.csv')
split['abs_path'] = split['video_path'].map(by_rel)
missing = split['abs_path'].isna().sum()
print('rows:', len(split), '| missing files:', missing)
assert missing == 0, split[split.abs_path.isna()].head()
split.to_csv(f'{ROOT}/splits/hifi_main_abs.csv', index=False)

## Verify the HAVIC checkpoint actually loads
The official script uses `strict=False`, which silently ignores mismatched keys (e.g. a `module.` prefix).
We check explicitly — a non-empty *missing* list means the baseline would be meaningless.

In [ ]:
import torch
use_src('/content/HAVIC')
from src.models.HAVIC import HAVIC_FT
sd = torch.load(W, map_location='cpu')
sd = sd.get('state_dict', sd) if isinstance(sd, dict) else sd
prefixes = {k.split('.')[0] for k in sd}
print('top-level key prefixes:', sorted(prefixes)[:15])
sd = {k.removeprefix('module.'): v for k, v in sd.items()}
m = HAVIC_FT()
missing, unexpected = m.load_state_dict(sd, strict=False)
print('missing:', len(missing), missing[:10])
print('unexpected:', len(unexpected), unexpected[:10])
assert not missing, 'HAVIC weights did not load cleanly'
del m, sd; torch.cuda.empty_cache()
use_src(TS)

## HAVIC zero-shot baseline (official script), in parts of 50 videos
Each part of 50 videos is saved to Drive as soon as it finishes (`results/havic_parts/`).
Finished parts are skipped on re-runs, so a disconnect loses at most the current part.
The last cell merges all saved parts into one table.

The official script runs face detection on CPU (~20 s per video). Notebook 01 computes the same
HAVIC score for every clip much faster, and the final ablation table uses that. So you can treat
this as a **reproduction check**: set `MAX_PARTS = 2` (100 videos, ~35 min) or run all 16 parts.

In [ ]:
SUBSET = 'test'     # 'test' (762 clips) or 'all' (3,810 clips)
PART_SIZE = 50   # 1 video at a time on the GPU (2 caused out-of-memory skips on long real videos)
MAX_PARTS = None    # e.g. 2 = only run 2 new parts this session; None = all remaining
VERBOSE = True      # one line per video

import re, subprocess, time, math
sub = (split if SUBSET == 'all' else split[split.split == 'test']).reset_index(drop=True)
PARTS_DIR = f'{ROOT}/results/havic_parts/{SUBSET}'
os.makedirs(PARTS_DIR, exist_ok=True)
n_parts = math.ceil(len(sub) / PART_SIZE)

W_CLEAN = f'{ROOT}/weights/best_ft_model_clean.pth'
if not os.path.exists(W_CLEAN):
    sd = torch.load(W, map_location='cpu'); sd = sd.get('state_dict', sd)
    torch.save({k.removeprefix('module.'): v for k, v in sd.items()}, W_CLEAN); del sd

result_re = re.compile(r'^\[\d+/\d+\] (.+?): overall_label (\d), overall_pred \[(-?[\d.e+-]+)\]')
env = dict(os.environ, TORCH_FORCE_NO_WEIGHTS_ONLY_LOAD='1', PYTHONUNBUFFERED='1')

def run_part(k):
    chunk = sub.iloc[k * PART_SIZE:(k + 1) * PART_SIZE]
    in_csv = f'/content/havic_in_{SUBSET}_{k:03d}.csv'
    out = f'{PARTS_DIR}/part_{k:03d}.csv'
    chunk[['abs_path', 'label']].rename(columns={'abs_path': 'video_path', 'label': 'overall_label'}).to_csv(in_csv, index=False)
    cmd = ['python', '-u', 'sliding_window_infer.py', '--csv_file_path', in_csv, '--save_csv_path', out + '.tmp',
           '--finetune_path', W_CLEAN, '--mode', 'evaluation', '--max_workers', '1', '--verbose']
    p = subprocess.Popen(cmd, cwd='/content/HAVIC/evaluation', env=env, text=True, bufsize=1,
                         stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
    done = skipped = correct = 0; t0 = time.time()
    for line in p.stdout:
        line = line.rstrip()
        if (not line or 'Audio successfully' in line or line.startswith('Sliding Window Inference')
                or 'video_batch' in line or line.endswith('.mp4:') or 'Warning' in line and 'UserWarning' in line
                or 'Load Weights Info' in line or 'detection model loaded' in line or line.startswith('According to')
                or 'torch.load' in line or 'SourceChange' in line or '_check_container_source' in line
                or 'timm' in line or line.startswith('ACC:') or line.startswith('Inference results')):
            continue
        m = result_re.match(line)
        if m:
            path, label, logit = m.group(1), int(m.group(2)), float(m.group(3))
            prob = 1 / (1 + math.exp(-logit)); ok = (prob >= 0.5) == (label == 1)
            done += 1; correct += ok
            if VERBOSE or done % 25 == 0:
                print(f'  [part {k+1}/{n_parts} | {done + skipped:2d}/{len(chunk)}] {"/".join(path.split("/")[-2:]):34s} '
                      f'true={"FAKE" if label else "REAL"} p(fake)={prob:.3f} {"ok" if ok else "XX"} | {(time.time()-t0)/60:4.1f} min')
        elif line.startswith('[Warning]') or line.startswith('[Error]'):
            skipped += 1
            print(f'  [part {k+1}/{n_parts} | {done + skipped:2d}/{len(chunk)}] SKIPPED: {line[:120]}')
        else:
            print('  ', line)
    p.wait()
    if p.returncode != 0 or not os.path.exists(out + '.tmp'):
        raise RuntimeError(f'part {k} failed (exit code {p.returncode}); re-run this cell to retry it')
    os.replace(out + '.tmp', out)
    print(f'SAVED part {k+1}/{n_parts}: {done} scored, {skipped} skipped, acc {correct/max(done,1):.1%} -> {out}')

# Parts saved with SKIPPED videos (GPU out-of-memory) can be redone: set REDO_PARTS_WITH_SKIPS = True
REDO_PARTS_WITH_SKIPS = False
def has_skips(k):
    f = f'{PARTS_DIR}/part_{k:03d}.csv'
    return os.path.exists(f) and len(pd.read_csv(f)) < len(sub.iloc[k * PART_SIZE:(k + 1) * PART_SIZE])
todo = [k for k in range(n_parts) if not os.path.exists(f'{PARTS_DIR}/part_{k:03d}.csv')
        or (REDO_PARTS_WITH_SKIPS and has_skips(k))]
print(f'{n_parts} parts of {PART_SIZE} | {n_parts - len(todo)} already saved | {len(todo)} to do')
for k in (todo if MAX_PARTS is None else todo[:MAX_PARTS]):
    run_part(k)

## Merge all saved parts into one result

In [ ]:
use_src(TS)
from src.utils.metrics import compute_metrics, to_markdown
parts = sorted(glob.glob(f'{PARTS_DIR}/part_*.csv'))
res = pd.concat([pd.read_csv(f) for f in parts], ignore_index=True)
res = res.merge(split[['abs_path', 'engine']], left_on='video_path', right_on='abs_path')
res.to_csv(f'{ROOT}/results/havic_zeroshot_{SUBSET}.csv', index=False)
print(f'{len(parts)}/{n_parts} parts merged | scored {len(res)} of {len(sub)} clips')
rows = {'HAVIC zero-shot (all engines)': compute_metrics(res.overall_label, res.overall_pred)}
for eng, g in res.groupby('engine'):
    rows[f'  {eng} ({len(g)} clips)'] = compute_metrics(g.overall_label, g.overall_pred)
table = to_markdown(rows)
print(table)
open(f'{ROOT}/results/havic_zeroshot_{SUBSET}.md', 'w').write(table)